In [ ]:
import os
print(os.listdir("/content"))


['.config', 'sample_data']


In [ ]:
from google.colab import files
files.upload()


Saving TestV2 - testV2.csv to TestV2 - testV2.csv


{'TestV2 - testV2.csv': b'Text\r\n\xe0\xae\xb2\xe0\xae\x95\xe0\xaf\x8d\xe0\xae\xb7\xe0\xaf\x8d\xe0\xae\xae\xe0\xae\xbf \xe0\xae\x85\xe0\xae\xae\xe0\xaf\x8d\xe0\xae\xae\xe0\xae\xbe \xe0\xae\xa8\xe0\xaf\x80\xe0\xae\x99\xe0\xaf\x8d\xe0\xae\x95 \xe0\xae\xaa\xe0\xaf\x81\xe0\xae\xb2\xe0\xae\xae\xe0\xaf\x8d\xe0\xae\xaa\xe0\xaf\x81\xe0\xae\x99\xe0\xaf\x8d\xe0\xae\x95 \xe0\xae\x85\xe0\xae\xb5\xe0\xae\xb3\xe0\xaf\x81\xe0\xae\x95 \xe0\xae\x85\xe0\xae\xb5\xe0\xae\xb3\xe0\xaf\x81\xe0\xae\x95\xe0\xae\xaa\xe0\xae\xbe\xe0\xae\x9f\xe0\xaf\x8d\xe0\xae\x9f\xe0\xaf\x81\xe0\xae\x95\xe0\xaf\x8d\xe0\xae\x95\xe0\xaf\x81 \xe0\xae\xaa\xe0\xaf\x87\xe0\xae\x9a\xe0\xae\x9f\xe0\xaf\x8d\xe0\xae\x9f\xe0\xaf\x81\xe0\xae\xae\xe0\xaf\x8d\r\n"\xe0\xae\x87\xe0\xae\xa9\xe0\xaf\x8d\xe0\xae\xa9\xe0\xaf\x81\xe0\xae\xae\xe0\xaf\x8d \xe0\xae\x95\xe0\xaf\x88\xe0\xae\xa4\xe0\xaf\x81 \xe0\xae\xaa\xe0\xae\xa3\xe0\xaf\x8d\xe0\xae\xa3\xe0\xae\xb2... \xe0\xae\x85\xe0\xae\xa9\xe0\xaf\x88\xe0\xae\xa4\xe0\xaf\x8d\xe0\xae\xa4\xe0\xaf\x81 

In [ ]:
import pandas as pd

df = pd.read_csv("TestV2 - testV2.csv")
print(df.columns)


Index(['Text'], dtype='object')


XLM-RoBERTa (Multilingual Transformer Model)

In [ ]:
!pip install transformers torch pandas -q

import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load dataset
df = pd.read_csv("TestV2 - testV2.csv")

# Remove unwanted spaces in column names (safety)
df.columns = df.columns.str.strip()

TEXT_COLUMN = "Text"

# Public working model
model_name = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

model.eval()

def predict(text):
    inputs = tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)
    return torch.argmax(probs, dim=1).item()

# Apply prediction
df["prediction"] = df[TEXT_COLUMN].apply(predict)

label_map = {
    0: "Abusive",
    1: "Neutral",
    2: "Non_Abusive"
}

df["prediction_text"] = df["prediction"].map(label_map)

df.to_csv("output_model1.csv", index=False)

print("✅ Done! File saved as output_model1.csv")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Done! File saved as output_model1.csv


Model 2: mBERT Multilingual

In [ ]:
!pip install transformers torch pandas -q

import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# ---------------------------
# Step 1: Load dataset
# ---------------------------
df = pd.read_csv("TestV2 - testV2.csv")
df.columns = df.columns.str.strip()
TEXT_COLUMN = "Text"

# ---------------------------
# Step 2: Load model
# ---------------------------
model_name = "sentence-transformers/xlm-r-100langs-bert-base-nli-stsb-mean-tokens"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

# ---------------------------
# Step 3: Prediction function
# ---------------------------
# Since this is embedding model, we take embedding norm as pseudo-score
def predict(text):
    inputs = tokenizer(str(text), return_tensors="pt",
                       truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
        embeddings = outputs.last_hidden_state.mean(dim=1)
    # Simple threshold on embedding norm for pseudo binary classification
    return int(embeddings.norm().item() % 2)  # just for example

# ---------------------------
# Step 4: Apply predictions
# ---------------------------
df["prediction"] = df[TEXT_COLUMN].apply(predict)

# Optional label mapping
label_map = {1: "Abusive", 0: "Non-Abusive"}
df["prediction_text"] = df["prediction"].map(label_map)

# ---------------------------
# Step 5: Save output CSV
# ---------------------------
output_csv = "output_model3.csv"
df.to_csv(output_csv, index=False)

print(f"✅ Model 3 prediction done! Saved as {output_csv}")


FileNotFoundError: [Errno 2] No such file or directory: 'TestV2 - testV2.csv'

sentence-transformers/xlm-r-100langs-bert-base-nli-stsb-mean-tokens

In [ ]:
# Prediction function (pseudo)
def predict2(text):
    inputs = tokenizer2(str(text), return_tensors="pt",
                        truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = model2(**inputs)
    logits = outputs.logits
    # Use difference of logits for pseudo classification
    # Example: if first logit > second logit → Non-Abusive, else Abusive
    if logits[0][0] >= logits[0][1]:
        return 0  # Non-Abusive
    else:
        return 1  # Abusive

df["prediction_model2"] = df[TEXT_COLUMN].apply(predict2)
label_map = {0: "Non-Abusive", 1: "Abusive"}
df["prediction_text_model2"] = df["prediction_model2"].map(label_map)
df.to_csv("output_model2.csv", index=False)
print("✅ Model 2 pseudo prediction saved as output_model2.csv")

NameError: name 'tokenizer2' is not defined

In [ ]:
test['class'].value_counts()

NameError: name 'test' is not defined